[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashakram05/ayeshaAkram-flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb)

# ML-10 — Content Action Playbook

This notebook is a research and decision-support prototype for Week 7.

It turns the validated FlyRank Lane 2 model output into a ranked content review queue and action playbook. The objective is to prioritize human review, not to automate article changes or claim causal impact.

Important provenance note:

- The source of truth is the validated Week 5 / Week 6 warehouse pipeline.
- The target remains `future_decline_proxy`.
- The historical training window is February decision snapshot → March future outcome.
- The scoring population is the March 31 decision snapshot.
- April outcomes are used only for retrospective evaluation or honest audit, not as model inputs for the March queue.
- Older starter/reference artifacts are not the Week 7 source of truth.

## 0. Colab-safe setup and honest methodology

This notebook is intended to run directly from GitHub in Google Colab. It therefore does the following before any modeling work:

- detects whether it is running in Colab
- clones the repository if needed
- changes to the repo root so relative paths work
- checks for the FlyRank Hugging Face token
- fails clearly if access is missing, instead of silently switching to stale starter files

The methodology is intentionally conservative and matches Weeks 5–6:

- Train on the February decision snapshot and label with March future performance.
- Score the March 31 decision snapshot using only decision-time features.
- Keep future outcome information separate from the scoring model.
- Use the Week 6 validation evidence to keep the final model choice consistent with the honest chronology.

This notebook is not production software and should not be presented as a final business system.

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# --- Detect repo root so the notebook works in Colab as well as local VS Code runs ---
if "google.colab" in sys.modules:
    REPO_URL = "https://github.com/ashakram05/ayeshaAkram-flyrank"
    REPO_ROOT = Path("/content/ayeshaAkram-flyrank")
    if not REPO_ROOT.exists():
        print(f"Colab detected. Cloning {REPO_URL} into {REPO_ROOT}...")
        !git clone --depth 1 {REPO_URL} {REPO_ROOT}
    os.chdir(REPO_ROOT)
    print(f"Repository root set to: {REPO_ROOT}")
else:
    here = Path.cwd()
    REPO_ROOT = None
    for candidate in [here, here.parent, here.parent.parent]:
        if (candidate / "work").exists() and (candidate / "skills").exists():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is None:
        raise FileNotFoundError(
            "Could not locate the FlyRank repo root. "
            "Run this notebook from the project root or open it from GitHub in Colab."
        )
    os.chdir(REPO_ROOT)
    print(f"Repository root set to: {REPO_ROOT}")

HF_TOKEN = (
    os.environ.get("HF_TOKEN")
    or os.environ.get("HUGGINGFACE_TOKEN")
    or os.environ.get("FLYRANK_HF_TOKEN")
)
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("flyrank")
    except Exception:
        HF_TOKEN = None

if not HF_TOKEN:
    raise RuntimeError(
        "FlyRank warehouse access is required for the real Week 7 queue. "
        "In Google Colab, store the Hugging Face token in Colab Secrets under the name 'flyrank' "
        "and run all cells again. Do not silently fall back to the old starter dataset or stale outputs."
    )

con = duckdb.connect()
con.execute(f"""
CREATE OR REPLACE SECRET flyrank_hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"
FEB_PATH = f"{REL}/fact_content_daily_performance/month=2026-02/*.parquet"
MAR_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
APR_PATH = f"{REL}/fact_content_daily_performance/month=2026-04/*.parquet"

FEATURE_COLS = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_ctr",
    "gsc_avg_position_clean",
    "has_position_data",
    "ga4_sessions_clean",
    "ga4_users_clean",
    "has_ga4",
]

FINAL_MODEL_NAME = "Week-5 Logistic Regression"
FINAL_MODEL = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

OUTPUT_DIR = REPO_ROOT / "work" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Warehouse access confirmed.")
print(f"Repo root: {REPO_ROOT}")
print(f"Training data path: {FEB_PATH}")
print(f"Scoring snapshot path: {MAR_PATH}")
print(f"Retrospective outcome path: {APR_PATH}")

Colab detected. Cloning https://github.com/ashakram05/ayeshaAkram-flyrank into /content/ayeshaAkram-flyrank...
Cloning into '/content/ayeshaAkram-flyrank'...
remote: Enumerating objects: 96, done.
remote: Counting objects: 100% (96/96), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 96 (delta 12), reused 72 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (96/96), 2.01 MiB | 7.26 MiB/s, done.
Resolving deltas: 100% (12/12), done.
Repository root set to: /content/ayeshaAkram-flyrank
Warehouse access confirmed.
Repo root: /content/ayeshaAkram-flyrank
Training data path: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet
Scoring snapshot path: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet
Retrospective outcome path: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet


## 1. Data source, train/test chronology, and honest labels

This notebook follows the Week 6 chronology exactly.

### Training data
- February decision snapshot
- March future outcome
- Label: `future_decline_proxy = 1` when future average daily impressions are below the decision-time impression level

### Scoring population
- March 31 decision snapshot
- No April outcome is used to train the model that scores those March rows
- April outcomes may be attached for retrospective evaluation only

### Why this matters
The operational queue is intended to prioritize human review. It is not a proof that a page should be refreshed, and it is not a causal statement about what will improve ranking performance.

In [5]:
def load_snapshot(path: str, decision_date: str) -> pd.DataFrame:
    return con.sql(f"""
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_data_available,
            ga4_data_available,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            ga4_sessions,
            ga4_users
        FROM read_parquet('{path}')
        WHERE report_date = DATE '{decision_date}'
    """).df()


def load_forward_outcome(path: str) -> pd.DataFrame:
    return con.sql(f"""
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS future_avg_daily_impressions,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS future_days_observed
        FROM read_parquet('{path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    """).df()


def attach_future_decline(snapshot: pd.DataFrame, future_outcome: pd.DataFrame) -> pd.DataFrame:
    out = snapshot.merge(future_outcome, on=["client_hash_id", "content_hash_id"], how="inner").copy()
    out["future_decline_proxy"] = (out["future_avg_daily_impressions"] < out["gsc_impressions"]).astype(int)
    return out


def prepare_features(frame: pd.DataFrame, position_median: float | None = None) -> pd.DataFrame:
    out = frame.copy()
    out["gsc_avg_position_clean"] = out["gsc_avg_position"].replace(0, np.nan)
    out["has_position_data"] = out["gsc_avg_position"].notna() & (out["gsc_avg_position"] != 0)

    if position_median is None:
        position_median = out["gsc_avg_position_clean"].median()
    out["gsc_avg_position_clean"] = out["gsc_avg_position_clean"].fillna(position_median)

    out["has_ga4"] = out["ga4_data_available"].astype(float).fillna(0).astype(int)
    out["ga4_sessions_clean"] = out["ga4_sessions"].where(out["ga4_data_available"] == True, 0).fillna(0)
    out["ga4_users_clean"] = out["ga4_users"].where(out["ga4_data_available"] == True, 0).fillna(0)
    out["gsc_ctr"] = np.where(out["gsc_impressions"] > 0, out["gsc_clicks"] / out["gsc_impressions"], 0.0)

    return out[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_impressions",
            "gsc_clicks",
            "gsc_ctr",
            "gsc_avg_position_clean",
            "has_position_data",
            "ga4_sessions_clean",
            "ga4_users_clean",
            "has_ga4",
            "future_decline_proxy",
        ]
    ].copy()


feb_snapshot = load_snapshot(FEB_PATH, "2026-02-28")
mar_snapshot = load_snapshot(MAR_PATH, "2026-03-31")
mar_outcome = load_forward_outcome(MAR_PATH)
apr_outcome = load_forward_outcome(APR_PATH)

train_raw = attach_future_decline(feb_snapshot, mar_outcome)
test_raw = attach_future_decline(mar_snapshot, apr_outcome)

position_median = train_raw["gsc_avg_position"].replace(0, np.nan).median()
train_df = prepare_features(train_raw, position_median)
test_df = prepare_features(test_raw, position_median)

print(f"Training rows (Feb -> Mar): {len(train_df):,}")
print(f"Scoring rows (Mar 31 snapshot): {len(test_df):,}")
print(f"Training base rate: {train_df['future_decline_proxy'].mean():.3f}")
print(f"Test base rate: {test_df['future_decline_proxy'].mean():.3f}")
print("Note: April outcomes are retained only for retrospective audit and are not used as features for the March scoring model.")

Training rows (Feb -> Mar): 150,802
Scoring rows (Mar 31 snapshot): 176,441
Training base rate: 0.221
Test base rate: 0.371
Note: April outcomes are retained only for retrospective audit and are not used as features for the March scoring model.


## 2. Final model choice and scoring logic

The Week 6 validation audit is the deciding evidence. The final model for the operational queue remains the Week-5 Logistic Regression.

This is not a new model choice. It is a continuation of the validated methodology:

- Train only on the February → March labeled data
- Evaluate using the honest chronology established in Week 6
- Use the March 31 decision-time features to score the queue
- Keep all future outcome information separate from the scoring model inputs

The queue is a human-review prioritization tool, not a production automation system.

In [6]:
def precision_at_k(y_true: pd.Series, scores: pd.Series, k: int) -> float:
    order = np.argsort(-np.asarray(scores), kind="stable")
    top_k = order[:k]
    return float(np.asarray(y_true)[top_k].mean())


def score_table(name: str, y_true: pd.Series, scores: pd.Series, rows: list[dict]) -> list[dict]:
    row = {"Method": name}
    for k in [10, 20, 50]:
        row[f"Precision@{k}"] = precision_at_k(y_true, scores, k)
    row["Base Rate"] = float(np.mean(y_true))
    rows.append(row)
    return rows


X_train = train_df[FEATURE_COLS]
y_train = train_df["future_decline_proxy"]
X_test = test_df[FEATURE_COLS]
y_test = test_df["future_decline_proxy"]

model = FINAL_MODEL
model.fit(X_train, y_train)

train_scores = model.predict_proba(X_train)[:, 1]
test_scores = model.predict_proba(X_test)[:, 1]

results = []
results = score_table("Week-5 Logistic Regression", y_test, test_scores, results)
print(pd.DataFrame(results).to_string(index=False))
print("Selected model: Week-5 Logistic Regression (consistent with the Week 6 honest validation evidence).")

scored_df = prepare_features(test_raw, position_median)
scored_df["model_probability"] = model.predict_proba(scored_df[FEATURE_COLS])[:, 1]
scored_df = scored_df.sort_values("model_probability", ascending=False).reset_index(drop=True)
scored_df["rank"] = np.arange(1, len(scored_df) + 1)

print(scored_df[["content_hash_id", "rank", "model_probability", "future_decline_proxy"]].head(10).to_string(index=False))

                    Method  Precision@10  Precision@20  Precision@50  Base Rate
Week-5 Logistic Regression           1.0          0.85          0.84   0.371172
Selected model: Week-5 Logistic Regression (consistent with the Week 6 honest validation evidence).
         content_hash_id  rank  model_probability  future_decline_proxy
content_fec55986a1868d62     1           1.000000                     1
content_0e03de7680314cd5     2           1.000000                     1
content_8e1334d6356668e3     3           1.000000                     1
content_eadb33b5df496f4a     4           1.000000                     1
content_046fc480045b88f5     5           0.999999                     1
content_8d7d99f109e19aa2     6           0.999999                     1
content_dc91779c3d085398     7           0.999999                     1
content_545bb6cc7081ded3     8           0.999997                     1
content_fa4cf3aa5ce67bb8     9           0.999986                     1
content_7172a7fad43f

## 3. Ranked actions, reason codes, archetype mapping, and human review rules

The operational queue is meant to support reviewer priority, not to trigger content changes automatically.

### Reason-code patterns
The reason codes below should be human-readable and tied to actual decision-time signals only, such as:

- low visibility
- weak engagement
- low CTR
- high opportunity signal
- position gap
- weak search quality signal

These are not a replacement for content judgment.

### Archetype → action mapping
The queue can suggest broad reviewer actions such as:

- review for freshness and factual correctness
- review for content expansion or refresh
- check metadata/search intent alignment
- monitor instead of refresh
- deprioritize if the signal is weak and business value is low

### Human review rules
The reviewer remains responsible for all decisions. The model may prioritize review, but it does not tell a human to publish, rewrite, delete, or make irreversible changes without review.

### No-go list
The system must not automatically:

- publish content
- delete or rewrite pages without human approval
- make legal, medical, or other high-stakes editorial claims
- treat model probability as ground truth
- make irreversible changes based on a ranking alone

### Cost/value thinking
Reviewer time is limited. The value of this queue comes from helping a person inspect the pages most likely to deserve attention first, while avoiding wasted effort on weak-signal or low-opportunity cases.

### Decay / refresh insight
The dataset can show an association between weaker recent performance and content that is older or lower-visibility. That supports a review hypothesis, but it is not proof that refreshes cause recovery. Observed patterns should be treated as decision-support signals, not causal claims.

In [7]:
def make_reason_codes(row: pd.Series) -> str:
    reasons = []
    if row["gsc_impressions"] < 500:
        reasons.append("low_visibility")
    if row["gsc_ctr"] < 0.01:
        reasons.append("low_ctr")
    if row["gsc_avg_position_clean"] > 20:
        reasons.append("position_gap")
    if row["ga4_sessions_clean"] < 5:
        reasons.append("weak_engagement")
    if not reasons:
        return "balanced_signal"
    return "|".join(reasons)


def infer_archetype(row: pd.Series) -> str:
    if row["gsc_impressions"] < 500 and row["ga4_sessions_clean"] < 5:
        return "low_visibility_low_engagement"
    if row["gsc_ctr"] < 0.01 and row["gsc_avg_position_clean"] > 20:
        return "search_visibility_gap"
    if row["ga4_sessions_clean"] >= 50 and row["gsc_impressions"] >= 1000:
        return "high_opportunity_content"
    return "mixed_signal_content"


def suggest_action(row: pd.Series) -> str:
    if row["model_probability"] >= 0.70 and row["gsc_ctr"] < 0.01:
        return "review_for_freshness_and_search_intent"
    if row["model_probability"] >= 0.60 and row["ga4_sessions_clean"] < 10:
        return "review_for_engagement_and_metadata"
    if row["model_probability"] >= 0.45:
        return "monitor_and_review"
    return "deprioritize_pending_human_check"


scored_df["reason_codes"] = scored_df.apply(make_reason_codes, axis=1)
scored_df["content_archetype"] = scored_df.apply(infer_archetype, axis=1)
scored_df["confidence"] = np.select(
    [scored_df["model_probability"] >= 0.70, scored_df["model_probability"] >= 0.45],
    ["high", "medium"],
    default="low",
)
scored_df["suggested_action"] = scored_df.apply(suggest_action, axis=1)

scored_df = scored_df.sort_values("model_probability", ascending=False).reset_index(drop=True)
scored_df["final_rank"] = np.arange(1, len(scored_df) + 1)

print("Top ranked review queue (decision-time signals only):")
print(
    scored_df[
        [
            "final_rank",
            "content_hash_id",
            "model_probability",
            "confidence",
            "content_archetype",
            "suggested_action",
            "reason_codes",
        ]
    ].head(10).to_string(index=False)
)

Top ranked review queue (decision-time signals only):
 final_rank          content_hash_id  model_probability confidence        content_archetype                       suggested_action            reason_codes
          1 content_fec55986a1868d62           1.000000       high     mixed_signal_content review_for_freshness_and_search_intent low_ctr|weak_engagement
          2 content_0e03de7680314cd5           1.000000       high     mixed_signal_content review_for_freshness_and_search_intent low_ctr|weak_engagement
          3 content_8e1334d6356668e3           1.000000       high     mixed_signal_content review_for_freshness_and_search_intent low_ctr|weak_engagement
          4 content_eadb33b5df496f4a           1.000000       high high_opportunity_content review_for_freshness_and_search_intent                 low_ctr
          5 content_046fc480045b88f5           0.999999       high     mixed_signal_content review_for_freshness_and_search_intent low_ctr|weak_engagement
          6 cont

## 4. Monitoring and retrain triggers

The queue should be monitored because a ranking can drift over time.

Proposed lightweight triggers include:

- the ranking performance shifts materially when new labels are produced
- the score distribution changes sharply from the previous queue
- feature definitions or data availability change
- a materially different content population appears
- enough new outcome data becomes available to run a fresh time-aware audit

These are operational reminders, not validated production thresholds.

In [8]:
queue_export = scored_df[
    [
        "final_rank",
        "content_hash_id",
        "model_probability",
        "confidence",
        "content_archetype",
        "suggested_action",
        "reason_codes",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_ctr",
        "gsc_avg_position_clean",
        "ga4_sessions_clean",
        "ga4_users_clean",
    ]
].copy()

queue_export = queue_export.rename(columns={"content_hash_id": "content_id"})
queue_export_path = OUTPUT_DIR / "week7_final_queue.csv"
summary_path = OUTPUT_DIR / "week7_queue_summary.json"

queue_export.to_csv(queue_export_path, index=False)

summary = {
    "final_model": FINAL_MODEL_NAME,
    "target": "future_decline_proxy",
    "training_window": "2026-02-28 decision snapshot -> 2026-03 future outcome",
    "scoring_window": "2026-03-31 decision snapshot",
    "queue_rows": int(len(queue_export)),
    "confidence_counts": queue_export["confidence"].value_counts().to_dict(),
    "action_counts": queue_export["suggested_action"].value_counts().to_dict(),
    "feature_set": FEATURE_COLS,
    "claim_language": "research and decision-support prototype; ranked review queue, not an automated production action system",
}
summary_path.write_text(__import__("json").dumps(summary, indent=2), encoding="utf-8")

print(f"Saved ranked queue to: {queue_export_path}")
print(f"Saved summary JSON to: {summary_path}")
print(summary)

Saved ranked queue to: /content/ayeshaAkram-flyrank/work/outputs/week7_final_queue.csv
Saved summary JSON to: /content/ayeshaAkram-flyrank/work/outputs/week7_queue_summary.json
{'final_model': 'Week-5 Logistic Regression', 'target': 'future_decline_proxy', 'training_window': '2026-02-28 decision snapshot -> 2026-03 future outcome', 'scoring_window': '2026-03-31 decision snapshot', 'queue_rows': 176441, 'confidence_counts': {'low': 169660, 'medium': 5831, 'high': 950}, 'action_counts': {'deprioritize_pending_human_check': 169660, 'monitor_and_review': 5086, 'review_for_freshness_and_search_intent': 949, 'review_for_engagement_and_metadata': 746}, 'feature_set': ['gsc_impressions', 'gsc_clicks', 'gsc_ctr', 'gsc_avg_position_clean', 'has_position_data', 'ga4_sessions_clean', 'ga4_users_clean', 'has_ga4'], 'claim_language': 'research and decision-support prototype; ranked review queue, not an automated production action system'}


## Self-check

- [x] Training uses the February decision snapshot and March outcome window.
- [x] The scoring pool is the March 31 decision snapshot with decision-time signals only.
- [x] The final model choice remains the Week-5 Logistic Regression, matching the Week 6 honest validation.
- [x] The queue is explicitly a human-review prioritization tool, not production automation.
- [x] The notebook fails clearly when required warehouse access is missing instead of silently falling back to stale outputs.
- [x] Ranked queue and summary outputs are written to `work/outputs/`.
